# Interactive Canny tuner

Use the sliders to tune the blur kernel and Canny thresholds, then click the save button to write the current edge map into the `cards` folder.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "ipywidgets is required for the interactive tuner. Install it in your notebook kernel with: pip install ipywidgets"
    ) from exc

dataset_dir = Path("iapr-26-uno-vision-challenge")
reference_path = Path("thresholded_refs") / "L1000768_yellow.png"
cards_dir = dataset_dir / "cards"
cards_dir.mkdir(exist_ok=True)

image_bgr = cv2.imread(str(reference_path))
if image_bgr is None:
    raise FileNotFoundError(f"Could not load {reference_path}")

gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
state = {"edges": None, "output_path": None}

blur_slider = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Blur")
low_slider = widgets.IntSlider(value=100, min=0, max=255, step=1, description="Low")
high_slider = widgets.IntSlider(value=200, min=0, max=255, step=1, description="High")
min_length_slider = widgets.IntSlider(value=0, min=0, max=1000, step=5, description="Min length")
save_button = widgets.Button(description="Save to cards", button_style="success")
status = widgets.Output()
preview = widgets.Output()

def filter_short_contours(edges, min_length: int):
    if min_length <= 0:
        return edges

    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    filtered = edges.copy()
    filtered[:] = 0

    for contour in contours:
        if cv2.arcLength(contour, closed=True) >= min_length:
            cv2.drawContours(filtered, [contour], -1, 255, thickness=1)

    return filtered

def compute_edges(blur_size: int, low_threshold: int, high_threshold: int, min_length: int):
    blurred = cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
    lower = min(low_threshold, high_threshold)
    upper = max(low_threshold, high_threshold)
    edges = cv2.Canny(blurred, threshold1=lower, threshold2=upper)
    return filter_short_contours(edges, min_length)

def update_preview(*_):
    edges = compute_edges(
        blur_slider.value,
        low_slider.value,
        high_slider.value,
        min_length_slider.value,
    )
    output_path = cards_dir / (
        f"{reference_path.stem}_canny_b{blur_slider.value}_l{min(low_slider.value, high_slider.value)}"
        f"_h{max(low_slider.value, high_slider.value)}_minlen{min_length_slider.value}.png"
    )
    state["edges"] = edges
    state["output_path"] = output_path

    with preview:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(edges, cmap="gray")
        ax.set_title(output_path.name)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    with status:
        clear_output(wait=True)
        print(f"Ready to save: {output_path}")

def save_edges(_):
    if state["edges"] is None or state["output_path"] is None:
        update_preview()

    cv2.imwrite(str(state["output_path"]), state["edges"])
    with status:
        clear_output(wait=True)
        print(f"Saved Canny edges to {state['output_path']}")

for slider in (blur_slider, low_slider, high_slider, min_length_slider):
    slider.observe(update_preview, names="value")

save_button.on_click(save_edges)

display(widgets.VBox([
    widgets.HBox([blur_slider, low_slider, high_slider, min_length_slider]),
    save_button,
    status,
    preview,
]))

update_preview()


In [10]:
import numpy as np

# Use the saved edge map as the template, or fall back to the current in-memory preview.
template_path = state["output_path"]
if template_path is not None and template_path.exists():
    template_edges = cv2.imread(str(template_path), cv2.IMREAD_GRAYSCALE)
elif state["edges"] is not None:
    template_edges = state["edges"].copy()
    template_path = reference_path
else:
    raise RuntimeError("Run the previous cell first so the edge template exists.")

# Pick a sample test image and compute its edge map with the current slider settings.
target_path = sorted((dataset_dir / "test_images").glob("*.jpg"))[0]
target_bgr = cv2.imread(str(target_path))
if target_bgr is None:
    raise FileNotFoundError(f"Could not load target image {target_path}")

target_gray_full = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2GRAY)
blurred_target = cv2.GaussianBlur(target_gray_full, (blur_slider.value, blur_slider.value), 0)
target_edges = cv2.Canny(
    blurred_target,
    threshold1=min(low_slider.value, high_slider.value),
    threshold2=max(low_slider.value, high_slider.value),
)
target_edges = filter_short_contours(target_edges, min_length_slider.value)

# Crop the template to its non-zero edge bounding box so we can reuse its aspect ratio.
ys, xs = np.where(template_edges > 0)
if len(xs) == 0 or len(ys) == 0:
    raise ValueError("The template edge map is empty.")
x0, x1 = xs.min(), xs.max()
y0, y1 = ys.min(), ys.max()
template_edges = template_edges[y0:y1 + 1, x0:x1 + 1]

# Downscale for speed on the high-resolution challenge images.
scale = 0.25
template_small = cv2.resize(template_edges, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
target_small = cv2.resize(target_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
target_edges_small = cv2.resize(target_edges, (target_small.shape[1], target_small.shape[0]), interpolation=cv2.INTER_NEAREST)

# Rectangular Hough: detect strong line segments, then look for contour quads.
line_canvas = cv2.cvtColor(target_edges_small, cv2.COLOR_GRAY2RGB)
lines = cv2.HoughLinesP(
    target_edges_small,
    rho=1,
    theta=np.pi / 180,
    threshold=60,
    minLineLength=40,
    maxLineGap=15,
)

line_mask = np.zeros_like(target_edges_small)
if lines is not None:
    for line in lines[:, 0]:
        x1, y1, x2, y2 = line
        cv2.line(line_canvas, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.line(line_mask, (x1, y1), (x2, y2), 255, 2)
else:
    raise RuntimeError("No Hough lines were detected.")

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
rect_mask = cv2.morphologyEx(line_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
contours, _ = cv2.findContours(rect_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

template_h, template_w = template_small.shape[:2]
template_ratio = template_w / template_h
candidates = []

for contour in contours:
    perimeter = cv2.arcLength(contour, True)
    if perimeter < 80:
        continue

    approx = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
    if len(approx) != 4 or not cv2.isContourConvex(approx):
        continue

    x, y, w, h = cv2.boundingRect(approx)
    if w < 30 or h < 30:
        continue

    area = cv2.contourArea(approx)
    rect_area = w * h
    fill_ratio = area / rect_area if rect_area > 0 else 0
    aspect_ratio = w / h
    ratio_error = abs(aspect_ratio - template_ratio) / template_ratio

    if fill_ratio < 0.5 or ratio_error > 0.5:
        continue

    score = perimeter * fill_ratio / (1 + ratio_error)
    candidates.append((score, approx, (x, y, w, h)))

if not candidates:
    raise RuntimeError("No rectangular candidates were found from the Hough lines.")

candidates.sort(key=lambda item: item[0], reverse=True)
top_k = min(10, len(candidates))
result_rgb = cv2.cvtColor(target_small, cv2.COLOR_BGR2RGB)

for rank, (_, approx, (x, y, w, h)) in enumerate(candidates[:top_k], start=1):
    cv2.polylines(result_rgb, [approx], True, (255, 0, 0), 2)
    cv2.putText(
        result_rgb,
        f"#{rank}",
        (x, max(20, y - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 0, 0),
        1,
        cv2.LINE_AA,
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(template_small, cmap="gray")
axes[0].set_title(f"Template: {template_path.name}")
axes[0].axis("off")

axes[1].imshow(line_canvas)
axes[1].set_title(f"Rectangular Hough lines in {target_path.name}")
axes[1].axis("off")

axes[2].imshow(result_rgb)
axes[2].set_title(f"Rectangle candidates ({top_k} best)")
axes[2].axis("off")

plt.tight_layout()
plt.show()


RuntimeError: No rectangular candidates were found from the Hough lines.